# 00 — Build benchmarks

Generates the Sudoku and HMC benchmarks, prints summary statistics, and saves figures to `thesis/figures/`.

## Setup

In [8]:
from pathlib import Path

from mnlearn.data.sudoku import (
    ensure_sudoku_csv,
    build_sudoku_benchmark,
    load_sudoku_benchmark,
)
from mnlearn.data.hmc import (
    build_hmc_benchmark,
    load_hmc_benchmark,
)
from mnlearn.data import figures, stats

SUDOKU_DIR = Path("../benchmarks/sudoku")
HMC_DIR    = Path("../benchmarks/hmc")
MNIST_ROOT = Path("../mnist_data")
THESIS_FIG = Path("../thesis/figures/data")
THESIS_FIG.mkdir(parents=True, exist_ok=True)

## Sudoku

### Build

In [9]:
csv_path = ensure_sudoku_csv()

build_sudoku_benchmark(
    output_dir=SUDOKU_DIR,
    modes=("symbolic", "visual"),
    csv_path=csv_path,
    num_puzzles=50_000,
    train_size=30_000, val_size=10_000, test_size=10_000,
    mnist_root=MNIST_ROOT,
)

data   = load_sudoku_benchmark(SUDOKU_DIR, mode="symbolic")
data_v = load_sudoku_benchmark(SUDOKU_DIR, mode="visual", mnist_root=MNIST_ROOT)

### Statistics

In [10]:
quizzes   = data["train"].quizzes.numpy()
solutions = data["train"].solutions.numpy()

clues = stats.sudoku_clues_per_puzzle(quizzes)
rate  = stats.sudoku_clue_rate_per_cell(quizzes)

print(f"blank rate:       {stats.sudoku_blank_rate(quizzes):.4f}")
print(f"clues per puzzle: mean={clues.mean():.1f}, std={clues.std():.1f}, min={clues.min()}, max={clues.max()}")
print(f"clue rate / cell: mean={rate.mean():.4f}, min={rate.min():.4f}, max={rate.max():.4f}")
print(f"solution digits:  {stats.sudoku_digit_distribution(solutions)}")

blank rate:       0.5825
clues per puzzle: mean=33.8, std=0.9, min=29, max=37
clue rate / cell: mean=0.4175, min=0.4006, max=0.4297
solution digits:  [     0 270000 270000 270000 270000 270000 270000 270000 270000 270000]


### Figures

In [11]:
N = len(data["train"])

figures.plot_sudoku_symbolic_example(
    data["train"].quizzes[0], data["train"].solutions[0],
    THESIS_FIG / "sudoku_symbolic_example.png",
)
figures.plot_sudoku_visual_example(
    data_v["train"], idx=0,
    save_path=THESIS_FIG / "sudoku_visual_example.png",
)
figures.plot_sudoku_difficulty(
    data["train"].quizzes,
    THESIS_FIG / "sudoku_difficulty.png",
    train_size=N,
)
figures.plot_sudoku_clue_rate_per_cell(
    data["train"].quizzes,
    THESIS_FIG / "sudoku_clue_rate_per_cell.png",
    train_size=N,
)
figures.plot_mnist_pool_usage_sudoku(
    data_v["train"].image_indices, data_v["train"].quizzes,
    THESIS_FIG / "sudoku_mnist_pool_usage.png",
    train_size=N,
)

WindowsPath('../thesis/figures/data/sudoku_mnist_pool_usage.png')

## HMC

### Build

In [12]:
build_hmc_benchmark(
    output_dir=HMC_DIR,
    modes=("symbolic", "visual"),
    num_samples=50_000,
    seq_len=30, num_states=10,
    p_self=0.7, p_emit=0.7,
    train_size=30_000, val_size=10_000, test_size=10_000,
    mnist_root=MNIST_ROOT,
)

data   = load_hmc_benchmark(HMC_DIR, mode="symbolic")
data_v = load_hmc_benchmark(HMC_DIR, mode="visual", mnist_root=MNIST_ROOT)

### Statistics

In [13]:
labels = data["train"].labels.numpy()
obs    = data["train"].obs_indices.numpy()
K      = data["config"]["num_states"]
p_self = data["config"]["p_self"]

runs = stats.hmc_run_length_distribution(labels)

print(f"state distribution: {stats.hmc_state_distribution(labels, K).round(3)}")
print(f"empirical p_self:   {stats.hmc_empirical_p_self(labels):.4f}")
print(f"empirical p_emit:   {stats.hmc_emission_accuracy(obs, labels):.4f}")
print(f"run length:         mean={runs.mean():.2f}, std={runs.std():.2f}, theoretical={1/(1-p_self):.2f}")

state distribution: [0.099 0.1   0.101 0.101 0.099 0.1   0.1   0.099 0.101 0.099]
empirical p_self:   0.7000
empirical p_emit:   0.7008
run length:         mean=3.09, std=2.52, theoretical=3.33


### Figures

In [14]:
N = len(data["train"])

figures.plot_hmc_symbolic_example(
    data["train"].obs_indices[0], data["train"].labels[0],
    THESIS_FIG / "hmc_symbolic_example.png",
    K=K,
)
figures.plot_hmc_visual_example(
    data_v["train"], idx=0,
    save_path=THESIS_FIG / "hmc_visual_example.png",
)
figures.plot_hmc_transition(
    data["train"].labels, K,
    THESIS_FIG / "hmc_transition.png",
    train_size=N,
)
figures.plot_mnist_pool_usage_hmc(
    data_v["train"].image_indices, data_v["train"].obs_indices, K,
    THESIS_FIG / "hmc_mnist_pool_usage.png",
    train_size=N,
)

WindowsPath('../thesis/figures/data/hmc_mnist_pool_usage.png')